In [1]:

import pandas as pd
import numpy as np
import pickle, json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                               GradientBoostingRegressor, GradientBoostingClassifier,
                               RandomForestRegressor)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, r2_score, mean_absolute_error, brier_score_loss

DATA_PATH = '../data/'
MODEL_DIR = Path('../backend/models/')
MODEL_DIR.mkdir(exist_ok=True)

DS1 = pd.read_excel(DATA_PATH + 'DS1_new_policy_registration_FIXED.xlsx')
DS2 = pd.read_excel(DATA_PATH + 'DS2_ml_training_FIXED.xlsx')
DS3 = pd.read_excel(DATA_PATH + 'DS3_policy_renewal_FIXED.xlsx')
DS4 = pd.read_excel(DATA_PATH + 'DS4_claims_FIXED.xlsx')
print(f'DS1={DS1.shape}  DS2={DS2.shape}  DS3={DS3.shape}  DS4={DS4.shape}')
print(f'DS2 accident rate: {DS2["Had_Accident"].mean():.1%}')


DS1=(33493, 35)  DS2=(28069, 24)  DS3=(21990, 28)  DS4=(33215, 21)
DS2 accident rate: 35.1%


7.1 Feature Engineering Transformer

In [3]:

class DerivedFeatureAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = pd.DataFrame(X).copy() if not isinstance(X, pd.DataFrame) else X.copy()
        X.columns = X.columns.astype(str)
        age = X.get('Driver_Age', pd.Series(35, index=X.index)).astype(float)
        exp_col = next((c for c in ['Years_Driving_Experience','Years_of_Driving_Experience']
                        if c in X.columns), None)
        exp = X[exp_col].astype(float) if exp_col else pd.Series(10, index=X.index)
        X['Experience_Rate']  = exp / (age - 17).clip(lower=1)
        X['Age_x_Exp']        = age * exp
        X['Is_Young_Driver']  = (age < 26).astype(int)
        X['Is_Senior_Driver'] = (age > 65).astype(int)
        X['Is_New_Driver']    = (exp < 3).astype(int)
        X['Is_Exp_Driver']    = (exp >= 10).astype(int)
        if 'Engine_CC' in X.columns and 'Vehicle_Age_Years' in X.columns:
            X['CC_x_VehicleAge'] = X['Engine_CC'].astype(float) * X['Vehicle_Age_Years'].astype(float)
            X['High_CC']         = (X['Engine_CC'].astype(float) > 2000).astype(int)
            X['Old_Vehicle']     = (X['Vehicle_Age_Years'].astype(float) > 10).astype(int)
        if 'Previous_NCB_Percentage' in X.columns:
            X['High_NCB'] = (X['Previous_NCB_Percentage'].astype(float) >= 30).astype(int)
        if 'Proposed_Sum_Insured_LKR' in X.columns and 'Market_Value_LKR' in X.columns:
            si = X['Proposed_Sum_Insured_LKR'].astype(float)
            mv = X['Market_Value_LKR'].astype(float).clip(lower=1)
            X['SI_MV_Ratio'] = si / mv
        return X.fillna(0)
print(" DerivedFeatureAdder defined")


 DerivedFeatureAdder defined


7.2 Build & Save Risk Classifier Pipeline

In [4]:

# ── Encode DS2 ────────────────────────────────────────────────────────
def encode_ds2(df):
    X = DerivedFeatureAdder().fit_transform(df.copy())
    encoders = {}
    for col in ['Gender','Vehicle_Type','Occupation','Province','Vehicle_Condition']:
        if col in X.columns:
            le = LabelEncoder()
            X[col+'_enc'] = le.fit_transform(X[col].astype(str))
            encoders[col] = le
    return X, encoders

DS2_fe, risk_encoders = encode_ds2(DS2)

RISK_FEATURES = [
    'Driver_Age','Years_Driving_Experience','Experience_Rate','Age_x_Exp',
    'Is_Young_Driver','Is_Senior_Driver','Is_New_Driver','Is_Exp_Driver',
    'Engine_CC','Vehicle_Age_Years','CC_x_VehicleAge','High_CC','Old_Vehicle',
    'Previous_NCB_Percentage','High_NCB',
    'Gender_enc','Vehicle_Type_enc','Occupation_enc','Province_enc',
]
if 'Vehicle_Condition_enc' in DS2_fe.columns:
    RISK_FEATURES.append('Vehicle_Condition_enc')

X_risk = DS2_fe[RISK_FEATURES].fillna(0)
y_risk = DS2_fe['Had_Accident']
print(f"Risk features: {len(RISK_FEATURES)}")
print(f"Class balance: {y_risk.value_counts().to_dict()}")

X_tr, X_te, y_tr, y_te = train_test_split(X_risk.values, y_risk.values,
                                            test_size=0.2, random_state=42, stratify=y_risk.values)

# Best model: HistGradientBoosting + Sigmoid calibration
risk_model = HistGradientBoostingClassifier(
    max_iter=400, max_depth=5, learning_rate=0.05,
    min_samples_leaf=20, l2_regularization=0.1,
    class_weight='balanced', random_state=42,
    early_stopping=True, validation_fraction=0.1, scoring='roc_auc'
)
risk_model.fit(X_tr, y_tr)
risk_cal = CalibratedClassifierCV(risk_model, method='sigmoid', cv=5)
risk_cal.fit(X_tr, y_tr)
risk_cal.fit(X_risk.values, y_risk.values)  # refit on all data

y_prob = risk_cal.predict_proba(X_te)[:,1]
auc    = roc_auc_score(y_te, y_prob)
brier  = brier_score_loss(y_te, y_prob)
print(f"\nRisk Classifier AUC:   {auc:.4f}")
print(f"Risk Classifier Brier: {brier:.4f}")
print(f"(Expected AUC 0.68-0.78 with deterministic targets)")


Risk features: 20
Class balance: {0: 18205, 1: 9864}

Risk Classifier AUC:   0.7471
Risk Classifier Brier: 0.1851
(Expected AUC 0.68-0.78 with deterministic targets)


7.3 Build & Save Premium Rate + Renewal Models

In [6]:

# ── Encode DS1 ─────────────────────────────────────────────────────
DS1_fe = DerivedFeatureAdder().fit_transform(DS1.copy())
prem_encoders = {}
for col in ['Gender','Vehicle_Type','Province','Occupation','Vehicle_Condition']:
    if col in DS1_fe.columns:
        le = LabelEncoder()
        DS1_fe[col+'_enc'] = le.fit_transform(DS1_fe[col].astype(str))
        prem_encoders[col] = le

def yn(x): return int(str(x).strip().lower() in ('yes','true','1'))
for c in ['Is_Existing_Customer','Is_Blacklisted','Rebate_Approved']:
    if c in DS1_fe.columns:
        DS1_fe[c] = DS1_fe[c].apply(yn)

if 'Proposed_Sum_Insured_LKR' in DS1_fe.columns and 'Market_Value_LKR' in DS1_fe.columns:
    DS1_fe['SI_MV_Ratio'] = DS1_fe['Proposed_Sum_Insured_LKR'] / DS1_fe['Market_Value_LKR'].clip(lower=1)

DS1_fe['rate_pct'] = DS1_fe['Net_Premium_LKR'] / DS1_fe['Proposed_Sum_Insured_LKR'].clip(lower=1)
DS1_fe = DS1_fe[(DS1_fe['rate_pct']>=0.015) & (DS1_fe['rate_pct']<=0.07)]

RATE_FEATURES = [c for c in [
    'Driver_Age','Years_of_Driving_Experience','Experience_Rate','Is_Young_Driver','Is_Senior_Driver',
    'Engine_CC','Vehicle_Age_Years','Vehicle_Type_enc','Province_enc','Occupation_enc',
    'NCB_Claimed_Percentage','Is_Blacklisted','Is_Existing_Customer','Risk_Score','SI_MV_Ratio','Rebate_Approved',
] if c in DS1_fe.columns]

y_rate = DS1_fe['rate_pct'].values
X_rate = DS1_fe[RATE_FEATURES].fillna(0).values

rate_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.04,
    subsample=0.8, random_state=42
)
rate_model.fit(X_rate, y_rate)
rate_r2 = r2_score(y_rate, rate_model.predict(X_rate))
print(f"Rate model train R²: {rate_r2:.4f}")

# ── Renewal model ─────────────────────────────────────────────────
DS3_fe = DS3.copy()
if 'Sum_Insured_Inline_Market' in DS3_fe.columns:
    DS3_fe['Sum_Insured_Inline_Market'] = (DS3_fe['Sum_Insured_Inline_Market'].astype(str).str.lower()=='yes').astype(int)
if 'Claim_Frequency_Pattern' in DS3_fe.columns:
    DS3_fe['Claim_Frequency_Pattern_enc'] = LabelEncoder().fit_transform(DS3_fe['Claim_Frequency_Pattern'].astype(str))

RENEW_FEATURES = [c for c in [
    'Previous_Premium_LKR','Previous_NCB_Percentage','New_NCB_Percentage',
    'Number_of_Claims','Total_Claim_Amount_Last_Year_LKR','Highest_Claim_Amount_LKR',
    'Days_Since_Last_Claim','Vehicle_Current_Age','Driver_Age','Years_With_Company',
    'Sum_Insured_Inline_Market','Claim_Frequency_Pattern_enc',
] if c in DS3_fe.columns]

y_ren = DS3_fe['Calculated_Renewal_Premium_LKR'].astype(float).values
X_ren = DS3_fe[RENEW_FEATURES].fillna(0).values

ren_model = HistGradientBoostingRegressor(
    max_iter=300, max_depth=5, learning_rate=0.05,
    l2_regularization=0.1, random_state=42
)
ren_model.fit(X_ren, y_ren)
ren_r2 = r2_score(y_ren, ren_model.predict(X_ren))
print(f"Renewal model train R²: {ren_r2:.4f}")


Rate model train R²: 0.9952
Renewal model train R²: 0.9901


7.4 Save pipeline_artifacts.pkl

In [8]:

artifacts = {
    # ── Risk classifier ──────────────────────────────────────────────
    'risk_model':           risk_cal,
    'risk_features':        RISK_FEATURES,
    'risk_auc':             auc,
    'risk_brier':           brier,
    # ── Premium rate model ────────────────────────────────────────────
    'rate_model':           rate_model,
    'rate_features':        RATE_FEATURES,
    'rate_r2':              rate_r2,
    # ── Renewal model ─────────────────────────────────────────────────
    'renewal_model':        ren_model,
    'renewal_features':     RENEW_FEATURES,
    'renewal_r2':           ren_r2,
    # ── Saved LabelEncoders (prevents training-serving skew) ──────────
    'encoders': {
        'risk':    risk_encoders,
        'premium': prem_encoders,
    },
    # ── Metadata ─────────────────────────────────────────────────────
    'trained_at':           datetime.now().isoformat(),
    'ds2_accident_rate':    float(y_risk.mean()),
    'n_training_policies':  len(DS1),
    'model_version':        '3.2-deterministic-targets',
    'notes': [
        'Had_Accident generated from deterministic risk function (not random)',
        'Risk_Score computed from features (not random normal)',
        'Premium predicts rate_pct (not raw premium) to avoid SI leakage',
        'Market_Value_LKR and Sum_Insured_LKR excluded from risk features',
    ]
}

pkl_path = MODEL_DIR / 'pipeline_artifacts.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(artifacts, f)
print(f" pipeline_artifacts.pkl saved → {pkl_path}")
print()
print("Contents:")
for k, v in artifacts.items():
    if k not in ('risk_model','rate_model','renewal_model','encoders'):
        print(f"  {k}: {v}")


 pipeline_artifacts.pkl saved → ..\backend\models\pipeline_artifacts.pkl

Contents:
  risk_features: ['Driver_Age', 'Years_Driving_Experience', 'Experience_Rate', 'Age_x_Exp', 'Is_Young_Driver', 'Is_Senior_Driver', 'Is_New_Driver', 'Is_Exp_Driver', 'Engine_CC', 'Vehicle_Age_Years', 'CC_x_VehicleAge', 'High_CC', 'Old_Vehicle', 'Previous_NCB_Percentage', 'High_NCB', 'Gender_enc', 'Vehicle_Type_enc', 'Occupation_enc', 'Province_enc', 'Vehicle_Condition_enc']
  risk_auc: 0.7471312457255621
  risk_brier: 0.1850937226383583
  rate_features: ['Driver_Age', 'Years_of_Driving_Experience', 'Experience_Rate', 'Is_Young_Driver', 'Is_Senior_Driver', 'Engine_CC', 'Vehicle_Age_Years', 'Vehicle_Type_enc', 'Province_enc', 'Occupation_enc', 'NCB_Claimed_Percentage', 'Is_Blacklisted', 'Is_Existing_Customer', 'Risk_Score', 'SI_MV_Ratio', 'Rebate_Approved']
  rate_r2: 0.995165573271004
  renewal_features: ['Previous_Premium_LKR', 'Previous_NCB_Percentage', 'New_NCB_Percentage', 'Number_of_Claims', 'Total_C